# OBD Soft-Prompt Training (OpenTSLM-SP)

Soft-prompt alignment using OpenTSLMSP with OBD-CoT dataset.

## 1) Setup

In [1]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (CWD, *CWD.parents) if (path / 'pyproject.toml').exists()), CWD)
for extra_path in (PROJECT_ROOT, PROJECT_ROOT / 'src', PROJECT_ROOT / 'open_flamingo', PROJECT_ROOT / 'src' / 'open_flamingo'):
    if extra_path.exists() and str(extra_path) not in sys.path:
        sys.path.insert(0, str(extra_path))

import json
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_
from transformers import get_linear_schedule_with_warmup

from opentslm.model.llm.OpenTSLMSP import OpenTSLMSP
from opentslm.system_metrics import SystemMetricsMonitor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = 'cuda'
# elif torch.backends.mps.is_available():
#     DEVICE = 'mps'
else:
    DEVICE = 'cpu'

DATA_PATH = PROJECT_ROOT / 'data/obd_cot_gpt5.jsonl'
LLM_ID = 'google/gemma-3-270m'
# LLM_ID = "meta-llama/Llama-3.2-1B"
BATCH_SIZE = 2
NUM_EPOCHS = 200
LR = 2e-4
WARMUP_FRAC = 0.1
GRAD_CLIP = 1.0
LLM_DTYPE = torch.bfloat16  # MPS-safe
patience = 3
METRICS_DIR = PROJECT_ROOT / 'results' / 'notebook_metrics'
METRICS_DIR.mkdir(parents=True, exist_ok=True)
RUN_SLUG = f"obd_soft_prompt_{LLM_ID.split('/')[-1].replace('-', '_').replace('.', '_').lower()}"


## 2) Load dataset

In [2]:
rows = [json.loads(x) for x in DATA_PATH.read_text(encoding='utf-8').splitlines() if x.strip()]
print(f'Amostras: {len(rows)}')
rows[0]

# Filter out 'congested' class
rows = [r for r in rows if r.get('label') != 'congested']
print('Amostras (sem congested):', len(rows))

# Rebuild pre_prompt to remove label leakage (keep two options, but no 'correct' hint)
_NO_LEAK_PROMPT = """You are shown a time-series plot of vehicle telemetry over a 120-sample window.
This data corresponds to one of two possible activities:
[{label}]
[{dis}]
Your task is to classify the activity based on analysis of the data.
Instructions:
- Begin by analyzing the time-series without assuming a specific label.
- Think step-by-step about what the observed patterns suggest regarding movement intensity and behavior.
- Write your rationale as a single, natural paragraph, do not use bullet points, numbered steps, or section headings.
- Do not refer back to the plot or to the act of visual analysis in your rationale; the plot is only for reference but you should reason about the time-series data.
- Do not assume any answer at the beginning, analyze as if you do not yet know which class is correct.
- Do not mention either class label until the final sentence.
- Make sure that your last word is the answer. You MUST end your response with \"Answer:\"."""

def _build_no_leak_prompt(r):
    label = r.get('label') or ''
    dis = r.get('dissimilar_label') or ''
    return _NO_LEAK_PROMPT.format(label=label, dis=dis)

for r in rows:
    r['pre_prompt'] = _build_no_leak_prompt(r)

# sanity check
print('pre_prompt sample (no leak):')
print(rows[0]['pre_prompt'])


Amostras: 195
Amostras (sem congested): 193
pre_prompt sample (no leak):
You are shown a time-series plot of vehicle telemetry over a 120-sample window.
This data corresponds to one of two possible activities:
[economical]
[aggressive]
Your task is to classify the activity based on analysis of the data.
Instructions:
- Begin by analyzing the time-series without assuming a specific label.
- Think step-by-step about what the observed patterns suggest regarding movement intensity and behavior.
- Write your rationale as a single, natural paragraph, do not use bullet points, numbered steps, or section headings.
- Do not refer back to the plot or to the act of visual analysis in your rationale; the plot is only for reference but you should reason about the time-series data.
- Do not assume any answer at the beginning, analyze as if you do not yet know which class is correct.
- Do not mention either class label until the final sentence.
- Make sure that your last word is the answer. You MUST 

## 3) Train/val/test split

In [3]:
idx = list(range(len(rows)))
random.shuffle(idx)
n = len(idx)
train_end = max(1, int(0.6*n))
val_end = max(train_end+1, int(0.8*n)) if n > 2 else train_end

train_rows = [rows[i] for i in idx[:train_end]]
val_rows = [rows[i] for i in idx[train_end:val_end]]
test_rows = [rows[i] for i in idx[val_end:]]

if len(val_rows) == 0 and len(train_rows) > 1:
    val_rows = [train_rows.pop()]
if len(test_rows) == 0 and len(train_rows) > 1:
    test_rows = [train_rows.pop()]

print('train/val/test:', len(train_rows), len(val_rows), len(test_rows))


train/val/test: 115 39 39


## 4) Dataset wrapper for OpenTSLM-SP

In [4]:
FEATURE_LABELS = [
    'Speed',
    'RPM',
    'Engine Load',
    'Throttle Position',
]

def summarize_series(label, values, n_points=8):
    values = values.detach().cpu().float()
    n = values.numel()
    idx = torch.linspace(0, max(n - 1, 0), steps=min(n_points, n)).long()
    samples = ', '.join(f'{values[i].item():.1f}' for i in idx)
    delta = values[-1].item() - values[0].item()
    if delta > 1e-3:
        trend = 'overall increasing'
    elif delta < -1e-3:
        trend = 'overall decreasing'
    else:
        trend = 'roughly flat'
    return (
        f'{label} series with mean {values.mean().item():.2f}, '
        f'std {values.std(unbiased=False).item():.2f}, '
        f'min {values.min().item():.2f}, max {values.max().item():.2f}, '
        f'trend {trend}, sampled points [{samples}]:'
    )

class OBDSPDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        ts = torch.tensor(r['time_series'], dtype=torch.float32)  # [F, T]
        # Expect 4 features: speed, rpm, load, throttle
        if ts.ndim != 2 or ts.shape[0] < 4:
            raise ValueError('time_series shape unexpected')
        # Build per-feature series and richer textual summaries.
        time_series = [ts[0], ts[1], ts[2], ts[3]]
        time_series_text = [
            summarize_series(label, series)
            for label, series in zip(FEATURE_LABELS, time_series)
        ]

        return {
            'pre_prompt': r['pre_prompt'],
            'time_series_text': time_series_text,
            'post_prompt': r['post_prompt'],
            'answer': r['answer'],
            'time_series': time_series,
            'id': r['id'],
        }

train_ds = OBDSPDataset(train_rows)
val_ds = OBDSPDataset(val_rows)
test_ds = OBDSPDataset(test_rows)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda x: x)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=lambda x: x)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=lambda x: x)


## 5) Initialize OpenTSLM-SP

In [5]:
model = OpenTSLMSP(llm_id=LLM_ID, device=DEVICE).to(DEVICE)

model.enable_lora(lora_r=16, lora_alpha=32, lora_dropout=0)
# keep fp32 on MPS
model.llm = model.llm.to(dtype=LLM_DTYPE)

trainable = [(n,p) for n,p in model.named_parameters() if p.requires_grad]
print('Trainable params:', sum(p.numel() for _,p in trainable))

optimizer = torch.optim.AdamW([p for _,p in trainable], lr=LR)
total_steps = max(1, NUM_EPOCHS * len(train_loader))
warmup_steps = int(WARMUP_FRAC * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


✅ LoRA enabled:
   LoRA parameters: 3,796,992
   Total trainable parameters: 3,796,992
   Total parameters: 271,895,808
   LoRA %: 1.40%
   Trainable %: 1.40%
Trainable params: 5990784


## 6) Sanity check

In [6]:
batch = next(iter(train_loader))
inputs_embeds, attention_mask = model.pad_and_apply_batch(batch)
print('inputs_embeds', inputs_embeds.shape)
print('attention_mask', attention_mask.shape)


inputs_embeds torch.Size([2, 660, 640])
attention_mask torch.Size([2, 660])


## 7) Train + validate

In [7]:
best_val = float('inf')
best_path = PROJECT_ROOT / 'data/obd_sp_best.pt'
no_improve = 0
train_metrics_path = METRICS_DIR / f'{RUN_SLUG}_train_samples.csv'
train_summary_path = METRICS_DIR / f'{RUN_SLUG}_train_summary.json'

train_monitor = SystemMetricsMonitor(
    label=f'{RUN_SLUG}_train',
    interval_s=0.5,
    disk_path=PROJECT_ROOT,
    metadata={'notebook': 'obd_soft_prompt_training', 'stage': 'train', 'llm_id': LLM_ID, 'device': DEVICE},
).start()
train_monitor.mark('setup', train_batches=len(train_loader), val_batches=len(val_loader), num_epochs=NUM_EPOCHS)

for epoch in range(1, NUM_EPOCHS+1):
    model.train()
    train_loss = 0.0
    batch_count = 0
    for step, batch in enumerate(train_loader, start=1):
        step_start = __import__('time').perf_counter()
        optimizer.zero_grad()
        loss = model.compute_loss(batch)
        if not torch.isfinite(loss):
            print('Loss not finite, aborting epoch.')
            train_monitor.mark('train_non_finite', epoch=epoch, step=step)
            break
        loss.backward()
        grad_norm = float(clip_grad_norm_(model.parameters(), GRAD_CLIP))
        optimizer.step()
        scheduler.step()
        loss_value = float(loss.item())
        train_loss += loss_value
        batch_count += 1
        train_monitor.mark(
            'train_step',
            epoch=epoch,
            step=step,
            loss=loss_value,
            grad_norm=grad_norm,
            lr=float(optimizer.param_groups[0]['lr']),
            step_duration_s=__import__('time').perf_counter() - step_start,
        )
    train_loss /= max(1, batch_count)

    model.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for val_step, batch in enumerate(val_loader, start=1):
            vloss = model.compute_loss(batch)
            if not torch.isfinite(vloss):
                print('Val loss not finite.')
                train_monitor.mark('val_non_finite', epoch=epoch, step=val_step)
                break
            val_loss += float(vloss.item())
            val_batches += 1
    val_loss /= max(1, val_batches)

    train_monitor.mark('epoch_end', epoch=epoch, train_loss=train_loss, val_loss=val_loss)
    print(f'Epoch {epoch}: train={train_loss:.6f} val={val_loss:.6f}')
    if val_loss < best_val:
        best_val = val_loss
        no_improve = 0
        model.store_to_file(str(best_path))
        train_monitor.mark('checkpoint_saved', epoch=epoch, best_val=best_val, checkpoint_path=str(best_path))
    else:
        no_improve += 1
        if no_improve >= patience:
            print('Early stopping acionado.')
            train_monitor.mark('early_stop', epoch=epoch, best_val=best_val)
            break

train_monitor.stop(final_phase='finished', best_val=best_val, checkpoint_path=str(best_path))
train_monitor.to_csv(train_metrics_path)
train_summary_path.write_text(json.dumps(train_monitor.summary().to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
print('Training metrics saved to', train_metrics_path)
print('Training summary saved to', train_summary_path)


Epoch 1: train=4.619218 val=4.288421
💾 Saved LoRA adapters with 252 parameters
Epoch 2: train=3.740049 val=3.322385
💾 Saved LoRA adapters with 252 parameters
Epoch 3: train=2.946981 val=2.819641
💾 Saved LoRA adapters with 252 parameters
Epoch 4: train=2.529945 val=2.567513
💾 Saved LoRA adapters with 252 parameters
Epoch 5: train=2.287342 val=2.422884
💾 Saved LoRA adapters with 252 parameters
Epoch 6: train=2.114464 val=2.343405
💾 Saved LoRA adapters with 252 parameters
Epoch 7: train=1.959614 val=2.281507
💾 Saved LoRA adapters with 252 parameters
Epoch 8: train=1.820902 val=2.274130
💾 Saved LoRA adapters with 252 parameters
Epoch 9: train=1.683825 val=2.257655
💾 Saved LoRA adapters with 252 parameters
Epoch 10: train=1.524854 val=2.292548
Epoch 11: train=1.384803 val=2.328134
Epoch 12: train=1.220522 val=2.433916
Early stopping acionado.
Training metrics saved to C:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-AutoTSLM\results\notebook_metrics\obd_soft_prompt_gemma_3_270m_train_samples.csv

## 8) Test inference

In [8]:
model.load_from_file(str(PROJECT_ROOT / 'data/obd_sp_best.pt'))
model.eval()
preds = []
inference_metrics_path = METRICS_DIR / f'{RUN_SLUG}_inference_samples.csv'
inference_summary_path = METRICS_DIR / f'{RUN_SLUG}_inference_summary.json'

inference_monitor = SystemMetricsMonitor(
    label=f'{RUN_SLUG}_inference',
    interval_s=0.5,
    disk_path=PROJECT_ROOT,
    metadata={'notebook': 'obd_soft_prompt_training', 'stage': 'inference', 'llm_id': LLM_ID, 'device': DEVICE},
).start()
inference_monitor.mark('setup', test_batches=len(test_loader))

with torch.no_grad():
    for step, batch in enumerate(test_loader, start=1):
        step_start = __import__('time').perf_counter()
        gen = model.generate(
            batch,
            max_new_tokens=256,
            eos_token_id=model.tokenizer.eos_token_id,
            pad_token_id=model.tokenizer.pad_token_id,
        )[0]
        latency = __import__('time').perf_counter() - step_start
        preds.append({
            'id': batch[0]['id'],
            'prediction': gen.strip(),
            'target': batch[0]['answer'],
        })
        inference_monitor.mark(
            'inference_step',
            step=step,
            inference_latency_s=latency,
            prediction_chars=len(gen.strip()),
            sample_id=batch[0]['id'],
        )

inference_monitor.stop(final_phase='finished', n_predictions=len(preds))
inference_monitor.to_csv(inference_metrics_path)
inference_summary_path.write_text(json.dumps(inference_monitor.summary().to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
print('Inference metrics saved to', inference_metrics_path)
print('Inference summary saved to', inference_summary_path)
preds[:3]


📥 Loaded LoRA adapters: 252 parameters
📥 Loaded model from epoch ?
Inference metrics saved to C:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-AutoTSLM\results\notebook_metrics\obd_soft_prompt_gemma_3_270m_inference_samples.csv
Inference summary saved to C:\Users\FUNPEC\Desktop\METROAUTOMOTIVE2026-AutoTSLM\results\notebook_metrics\obd_soft_prompt_gemma_3_270m_inference_summary.json


[{'id': 'obd_cot_000088',
  'prediction': 'coordinate coordinate 周期 TS_WINDOWUP 時間-序列 120:00 - 15:00 UTC 時-峰脈 TS_HANGAR Time-period: 120:00-15:00 Latent-period: 120:00 Latent: high window: 2; velocity: 122.81; temperature: 23.42; spectral index: 16.13; slope: -1.91; endplate: 120:00; endplate: 15:01 UTC PV_HANGAR Time-period: 120:00-15:00 Latent-period: 120:00 Latent: high window: 2; velocity: 145.77; temperature: 31.24; spectral index: 22.45; slope: -6.62; endplate: 120:00; endplate: 15:02 UTC GRB_ON SHIPPING Time-period: 588:00 - 11:00',
  'target': 'Speed stays mostly low with gradual ramps and plateaus, peaking only in the mid‑20s to around 30 km/h and spending considerable time near idle, indicating mild acceleration and frequent coasting. RPM largely tracks these modest changes, hovering near idle with brief rises below about 2000, never sustaining high revs that would suggest hard accelerations. Engine load remains in a moderate band, with smooth transitions rather than sharp su

In [9]:
import re
from difflib import SequenceMatcher
import numpy as np

def norm_text(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def tok(s: str):
    return re.findall(r"\w+", s.lower())

def token_f1(pred: str, gold: str) -> float:
    p = tok(pred)
    g = tok(gold)
    if not p or not g:
        return 0.0
    inter = 0
    g_used = [False] * len(g)
    for t in p:
        for i, gt in enumerate(g):
            if not g_used[i] and t == gt:
                inter += 1
                g_used[i] = True
                break
    prec = inter / len(p)
    rec = inter / len(g)
    return 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)

def rouge_l_f1(pred: str, gold: str) -> float:
    p = tok(pred)
    g = tok(gold)
    if not p or not g:
        return 0.0
    dp = [[0] * (len(g) + 1) for _ in range(len(p) + 1)]
    for i in range(1, len(p) + 1):
        for j in range(1, len(g) + 1):
            if p[i - 1] == g[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    lcs = dp[-1][-1]
    prec = lcs / len(p)
    rec = lcs / len(g)
    return 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)

def exact_match(pred: str, gold: str) -> float:
    return 1.0 if norm_text(pred) == norm_text(gold) else 0.0

def seq_sim(pred: str, gold: str) -> float:
    return SequenceMatcher(None, norm_text(pred), norm_text(gold)).ratio()

scores = {
    "token_f1": [],
    "rouge_l_f1": [],
    "exact_match": [],
    "seq_sim": [],
}

for x in preds:
    scores["token_f1"].append(token_f1(x["prediction"], x["target"]))
    scores["rouge_l_f1"].append(rouge_l_f1(x["prediction"], x["target"]))
    scores["exact_match"].append(exact_match(x["prediction"], x["target"]))
    scores["seq_sim"].append(seq_sim(x["prediction"], x["target"]))

print("Token-F1 medio:", round(float(np.mean(scores["token_f1"])) if scores["token_f1"] else 0.0, 4))
print("ROUGE-L F1 medio:", round(float(np.mean(scores["rouge_l_f1"])) if scores["rouge_l_f1"] else 0.0, 4))
print("Exact-Match medio:", round(float(np.mean(scores["exact_match"])) if scores["exact_match"] else 0.0, 4))
print("SeqSim medio:", round(float(np.mean(scores["seq_sim"])) if scores["seq_sim"] else 0.0, 4))

# Classification metrics (paper-style)
LABELS = ['economical', 'normal', 'aggressive', 'congested']

def extract_label(text):
    if text is None:
        return None
    t = str(text).lower()
    # Prefer explicit Answer: tag
    m = re.findall(r'answer\s*:\s*([a-z_\-]+)', t)
    if m:
        cand = m[-1]
        return cand if cand in LABELS else None
    # Fallback: last label occurrence
    positions = [(t.rfind(lbl), lbl) for lbl in LABELS if t.rfind(lbl) != -1]
    if positions:
        positions.sort()
        return positions[-1][1]
    return None

y_true = []
y_pred = []
for x in preds:
    gt = extract_label(x.get('target'))
    pr = extract_label(x.get('prediction'))
    if gt is None or pr is None:
        continue
    y_true.append(gt)
    y_pred.append(pr)

total = len(y_true)
correct = sum(1 for g,p in zip(y_true, y_pred) if g == p)
accuracy = (correct / total) if total else 0.0

# Macro-F1 over fixed label set
macro_f1 = 0.0
for lbl in LABELS:
    tp = sum(1 for g,p in zip(y_true, y_pred) if g == lbl and p == lbl)
    fp = sum(1 for g,p in zip(y_true, y_pred) if g != lbl and p == lbl)
    fn = sum(1 for g,p in zip(y_true, y_pred) if g == lbl and p != lbl)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)
    macro_f1 += f1
macro_f1 = macro_f1 / len(LABELS) if LABELS else 0.0

print('Accuracy:', round(float(accuracy), 4), f'({correct}/{total})')
print('Macro-F1:', round(float(macro_f1), 4))

# Supplementary semantic and structural metrics for paper reporting.
validity_rows = []
for x in preds:
    pred = str(x.get('prediction', ''))
    pred_label = extract_label(pred)
    has_answer_tag = bool(re.search(r'answer\s*:', pred.lower()))
    non_ascii_ratio = sum(ord(ch) > 127 for ch in pred) / max(len(pred), 1)
    validity_rows.append({
        'id': x.get('id'),
        'has_answer_tag': has_answer_tag,
        'pred_label': pred_label,
        'valid_label': pred_label in LABELS,
        'non_ascii_ratio': float(non_ascii_ratio),
        'prediction_length': len(pred),
    })

validity_metrics = {
    'has_answer_tag_rate': float(np.mean([r['has_answer_tag'] for r in validity_rows])) if validity_rows else 0.0,
    'valid_label_rate': float(np.mean([r['valid_label'] for r in validity_rows])) if validity_rows else 0.0,
    'mean_non_ascii_ratio': float(np.mean([r['non_ascii_ratio'] for r in validity_rows])) if validity_rows else 0.0,
    'mean_prediction_length': float(np.mean([r['prediction_length'] for r in validity_rows])) if validity_rows else 0.0,
}

print('Has Answer tag rate:', round(validity_metrics['has_answer_tag_rate'], 4))
print('Valid label rate:', round(validity_metrics['valid_label_rate'], 4))
print('Mean non-ASCII ratio:', round(validity_metrics['mean_non_ascii_ratio'], 4))

detailed_rows = []
for i, x in enumerate(preds):
    row = dict(x)
    row['token_f1'] = float(scores['token_f1'][i])
    row['rouge_l_f1'] = float(scores['rouge_l_f1'][i])
    row['exact_match'] = float(scores['exact_match'][i])
    row['seq_sim'] = float(scores['seq_sim'][i])
    row.update(validity_rows[i])
    detailed_rows.append(row)

advanced_metrics = {}
try:
    from bert_score import score as bertscore_score
    _, _, bert_f1 = bertscore_score(
        [x['prediction'] for x in preds],
        [x['target'] for x in preds],
        lang='en',
        verbose=False,
    )
    bert_f1 = bert_f1.detach().cpu().numpy().tolist()
    advanced_metrics['bertscore_f1_mean'] = float(np.mean(bert_f1)) if bert_f1 else 0.0
    for row, val in zip(detailed_rows, bert_f1):
        row['bertscore_f1'] = float(val)
    print('BERTScore F1 mean:', round(advanced_metrics['bertscore_f1_mean'], 4))
except Exception as exc:
    advanced_metrics['bertscore_f1_mean'] = None
    advanced_metrics['bertscore_error'] = str(exc)
    print('BERTScore unavailable:', exc)

try:
    from sentence_transformers import SentenceTransformer
    emb_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')
    pred_emb = emb_model.encode([x['prediction'] for x in preds], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    tgt_emb = emb_model.encode([x['target'] for x in preds], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    cosine_scores = (pred_emb * tgt_emb).sum(axis=1).tolist()
    advanced_metrics['embedding_cosine_mean'] = float(np.mean(cosine_scores)) if cosine_scores else 0.0
    for row, val in zip(detailed_rows, cosine_scores):
        row['embedding_cosine'] = float(val)
    print('Embedding cosine mean:', round(advanced_metrics['embedding_cosine_mean'], 4))
except Exception as exc:
    advanced_metrics['embedding_cosine_mean'] = None
    advanced_metrics['embedding_cosine_error'] = str(exc)
    print('Embedding cosine unavailable:', exc)


Token-F1 medio: 0.1311
ROUGE-L F1 medio: 0.0627
Exact-Match medio: 0.0
SeqSim medio: 0.0142
Accuracy: 0.8571 (6/7)
Macro-F1: 0.4643
Has Answer tag rate: 0.1282
Valid label rate: 0.1795
Mean non-ASCII ratio: 0.0784


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore F1 mean: 0.7651


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding cosine mean: 0.2638


In [10]:
# preds deve existir: lista de dicts com keys: id, prediction, target (e opcionalmente prompt)
# scores deve existir: dict com arrays (token_f1, rouge_l_f1, exact_match, seq_sim)

if LLM_ID == "google/gemma-3-270m":
    model_sufix = "gemma3_270m"
elif LLM_ID == "meta-llama/Llama-3.2-1B":
    model_sufix = "llama3_2_1B"

out_pred = Path(f"data/obd_eval_predictions_{model_sufix}.jsonl")
out_detail = Path(f"data/obd_eval_detailed_{model_sufix}.jsonl")
out_summary = Path(f"data/obd_eval_summary_{model_sufix}.json")

# salvar previsões
with out_pred.open("w", encoding="utf-8") as f:
    for p in preds:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

# salvar métricas por exemplo
with out_detail.open("w", encoding="utf-8") as f:
    for row in detailed_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# salvar resumo
summary = {
    "n": len(preds),
    "token_f1_mean": float(np.mean(scores["token_f1"])) if scores["token_f1"] else 0.0,
    "rouge_l_f1_mean": float(np.mean(scores["rouge_l_f1"])) if scores["rouge_l_f1"] else 0.0,
    "exact_match_mean": float(np.mean(scores["exact_match"])) if scores["exact_match"] else 0.0,
    "seq_sim_mean": float(np.mean(scores["seq_sim"])) if scores["seq_sim"] else 0.0,
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "validity_metrics": validity_metrics,
    "advanced_metrics": advanced_metrics,
}

out_summary.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved predictions to", out_pred)
print("Saved detailed metrics to", out_detail)
print("Saved summary to", out_summary)


Saved predictions to data\obd_eval_predictions_gemma3_270m.jsonl
Saved detailed metrics to data\obd_eval_detailed_gemma3_270m.jsonl
Saved summary to data\obd_eval_summary_gemma3_270m.json
